# Notebook 2: SFT 监督微调训练

本 Notebook 完成以下工作:
1. 使用 Unsloth 加载 Qwen 模型 (NF4 量化)
2. 配置 LoRA 参数
3. 加载 Notebook 1 格式化的 ChatML 数据
4. 执行 SFT 训练
5. 保存 LoRA Adapter + 合并完整模型
6. 验证模型输出质量

**技术栈**:
- Unsloth (QLoRA 加速，比原生 HuggingFace 快 2x)
- bitsandbytes NF4 量化 (显存优化)
- TRL SFTTrainer (标准化训练循环)

**硬件需求**: 1× A5000 (24GB), ~15GB VRAM
**预计耗时**: 3-6 小时 (100K 样本, 3 epochs)

In [ ]:
# ============================================================
# 0. 环境初始化
# ============================================================

import os
import sys
from pathlib import Path
import yaml
import torch

# 项目根目录
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

print(f" Project Root: {PROJECT_ROOT}")
print(f" CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")
    print(f" VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# ============================================================
# 1. 加载配置
# ============================================================

with open(PROJECT_ROOT / "config" / "sft.yaml", "r") as f:
    CONFIG = yaml.safe_load(f)

print("=== SFT Configuration ===")
for key, val in CONFIG.items():
    print(f"  {key}: {val}")

In [ ]:
# ============================================================
# 2. 加载模型 (Unsloth + NF4 量化)
# ============================================================

from unsloth import FastLanguageModel

model_cfg = CONFIG["model"]
lora_cfg = CONFIG["lora"]

print(f" Loading model: {model_cfg['name']}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_cfg["name"],
    max_seq_length=model_cfg["max_seq_length"],
    dtype=model_cfg["dtype"],
    load_in_4bit=model_cfg["load_in_4bit"],
    # token="hf_..."  # 如需访问 gated model，取消注释并填入 token
)

print(f" Model loaded!")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

In [ ]:
# ============================================================
# 3. 配置 LoRA
# ============================================================

print(" Configuring LoRA...")

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_cfg["r"],
    target_modules=lora_cfg["target_modules"],
    lora_alpha=lora_cfg["lora_alpha"],
    lora_dropout=lora_cfg["lora_dropout"],
    bias=lora_cfg["bias"],
    use_gradient_checkpointing=lora_cfg["use_gradient_checkpointing"],
    random_state=lora_cfg["random_state"],
    use_rslora=lora_cfg["use_rslora"],
    loftq_config=lora_cfg["loftq_config"],
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f" LoRA configured!")
print(f"  Trainable: {trainable_params/1e6:.1f}M params ({trainable_params/total_params*100:.2f}%)")
print(f"  Total:     {total_params/1e9:.1f}B params")

In [ ]:
# ============================================================
# 4. 加载格式化数据
# ============================================================

from datasets import load_from_disk
from utils.schema import SFTDataFormatter

SFT_DATA_DIR = PROJECT_ROOT / "data" / "sft_formatted"

# 检查数据是否已准备好
if not SFT_DATA_DIR.exists():
    print(f" ❌ SFT data not found at {SFT_DATA_DIR}!")
    print("    Please run Notebook 1 first.")
    # 使用 fallback: 直接从 HuggingFace 加载并格式化
    print("    Loading directly from HuggingFace as fallback...")
    from datasets import load_dataset
    
    ds = load_dataset("R6410418/Chinese-Qwen3-235B-Thinking-2507-Distill-100k", split="train")
    ds = ds.select(range(min(5000, len(ds))))  # 仅取 5000 条做 demo
    
    formatter = SFTDataFormatter(max_seq_length=model_cfg["max_seq_length"])
    
    def format_fn(example):
        return {"text": formatter.format(example)}
    
    dataset = ds.map(format_fn)
    dataset = dataset.train_test_split(test_size=0.05, seed=3407)
else:
    print(f" Loading SFT data from {SFT_DATA_DIR}...")
    dataset = load_from_disk(str(SFT_DATA_DIR))

print(f" Train: {len(dataset['train'])} samples")
print(f" Validation: {len(dataset['validation'])} samples")

In [ ]:
# ============================================================
# 5. 设置训练参数
# ============================================================

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

train_cfg = CONFIG["training"]

training_args = TrainingArguments(
    # 输出
    output_dir=train_cfg["output_dir"],
    
    # Batch
    per_device_train_batch_size=train_cfg["per_device_train_batch_size"],
    per_device_eval_batch_size=train_cfg["per_device_eval_batch_size"],
    gradient_accumulation_steps=train_cfg["gradient_accumulation_steps"],
    
    # Epochs
    num_train_epochs=train_cfg["num_train_epochs"],
    
    # 学习率
    learning_rate=train_cfg["learning_rate"],
    lr_scheduler_type=train_cfg["lr_scheduler_type"],
    warmup_ratio=train_cfg["warmup_ratio"],
    
    # 优化器
    optim=train_cfg["optim"],
    weight_decay=train_cfg["weight_decay"],
    max_grad_norm=train_cfg["max_grad_norm"],
    
    # 日志与保存
    logging_steps=train_cfg["logging_steps"],
    save_steps=train_cfg["save_steps"],
    eval_steps=train_cfg["eval_steps"],
    save_total_limit=train_cfg["save_total_limit"],
    save_strategy=train_cfg["save_strategy"],
    evaluation_strategy=train_cfg["evaluation_strategy"],
    load_best_model_at_end=train_cfg["load_best_model_at_end"],
    metric_for_best_model=train_cfg["metric_for_best_model"],
    
    # 精度
    fp16=train_cfg["fp16"],
    bf16=train_cfg["bf16"],
    
    # 其他
    seed=train_cfg["seed"],
    report_to=train_cfg.get("report_to", "none"),
    run_name=train_cfg.get("run_name", "sft_run"),
    
    # 避免 wandb 警告
    disable_tqdm=False,
)

print(" Training Arguments:")
print(f"  output_dir: {training_args.output_dir}")
print(f"  batch_size: {training_args.per_device_train_batch_size} × {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  epochs: {training_args.num_train_epochs}")
print(f"  learning_rate: {training_args.learning_rate}")
print(f"  fp16: {training_args.fp16}, bf16: {training_args.bf16}")

In [ ]:
# ============================================================
# 6. 创建 Trainer 并开始训练
# ============================================================

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    dataset_text_field="text",
    max_seq_length=model_cfg["max_seq_length"],
    dataset_num_proc=2,
    packing=False,  # Qwen 序列建议不 packing
)

print("\n Starting SFT training...")
print("=" * 60)

# 开始训练
trainer.train()

print("\n" + "=" * 60)
print(" Training complete!")

In [ ]:
# ============================================================
# 7. 保存模型
# ============================================================

import shutil

OUTPUT_DIR = Path(train_cfg["output_dir"])
FINAL_DIR = OUTPUT_DIR / "final_checkpoint"

# 7a. 保存 LoRA Adapter (轻量级，~50MB)
print(" Saving LoRA adapter...")
model.save_pretrained(str(FINAL_DIR / "lora_adapter"))
tokenizer.save_pretrained(str(FINAL_DIR / "lora_adapter"))
print(f"   → {FINAL_DIR / 'lora_adapter'}")

# 7b. 合并并保存完整模型 (用于 GGUF 导出)
print("\n Merging LoRA into base model (this uses more VRAM)...")
try:
    merged_model = model.merge_and_unload()
    merged_dir = FINAL_DIR / "merged_model"
    merged_model.save_pretrained(str(merged_dir))
    tokenizer.save_pretrained(str(merged_dir))
    print(f"   → {merged_dir}")
    
    # 释放合并模型的显存
    del merged_model
    torch.cuda.empty_cache()
except Exception as e:
    print(f"   ⚠️ Merge failed: {e}")
    print(f"   You can merge later with:")
    print(f"   model = FastLanguageModel.from_pretrained(...)")
    print(f"   model = model.merge_and_unload()")

# 7c. 保存训练 metrics
print("\n Saving training metrics...")
if hasattr(trainer, 'state'):
    import json as json_module
    log_history = trainer.state.log_history
    with open(FINAL_DIR / "training_metrics.json", "w") as f:
        json_module.dump(log_history, f, indent=2)
    print(f"   → {FINAL_DIR / 'training_metrics.json'}")

print(f"\n All artifacts saved to: {FINAL_DIR}")

In [ ]:
# ============================================================
# 8. 推理验证
# ============================================================

from unsloth import FastLanguageModel

# 切换到推理模式
FastLanguageModel.for_inference(model)

test_prompts = [
    "请解释什么是机器学习中的过拟合，并给出解决方案。",
    "计算: 如果一个正方形的面积是 64 平方厘米，它的边长是多少？",
    "What is the difference between supervised and unsupervised learning?",
]

print("=== Inference Tests ===\n")

for i, prompt in enumerate(test_prompts):
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant. Think step by step and put your final answer in <answer>...</answer> tags."},
        {"role": "user", "content": prompt},
    ]
    
    # ChatML 格式化
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize
    inputs = tokenizer(formatted, return_tensors="pt").to("cuda")
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    print(f"--- Test {i+1} ---")
    print(f"Prompt: {prompt}")
    print(f"Response: {response_text[:300]}...")
    print()

In [ ]:
# ============================================================
# 9. 训练总结
# ============================================================

print("=" * 60)
print("  SFT TRAINING SUMMARY")
print("=" * 60)
print(f"\n  Model: {model_cfg['name']}")
print(f"  LoRA: r={lora_cfg['r']}, alpha={lora_cfg['lora_alpha']}")
print(f"  Train samples: {len(dataset['train'])}")
print(f"  Epochs: {train_cfg['num_train_epochs']}")
print(f"  Learning rate: {train_cfg['learning_rate']}")
print(f"\n  Output: {OUTPUT_DIR}")
print(f"  LoRA adapter: {FINAL_DIR / 'lora_adapter'}")
print(f"  Merged model: {FINAL_DIR / 'merged_model'}")
print("\n" + "=" * 60)
print("  ✅ SFT complete! Proceed to Notebook 3 for RL training.")
print("=" * 60)